In [ ]:
import os
import re
import gc
import torch
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/SARVAM_RAG_EVALUATION"
os.makedirs(BASE_DIR, exist_ok=True)

print(BASE_DIR)

Mounted at /content/drive
/content/drive/MyDrive/SARVAM_RAG_EVALUATION


In [ ]:
!pip install -q rouge-score sacrebleu bert-score sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import torch
import pandas as pd
import numpy as np

from google.colab import drive
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

In [ ]:
CSV_PATH = "/content/drive/MyDrive/MALAYALAM_RAG_PROJECT/sarvam_rag/predictions.csv"

df = pd.read_csv(CSV_PATH)

print(df.columns)
print("Rows:", len(df))

df.head()

Index(['article', 'reference', 'prediction', 'contexts'], dtype='object')
Rows: 970


,article,reference,prediction,contexts
0,വയനാട്ടിൽ കാരാപ്പുഴ റിസർവോയർ പദ്ധതിക്കായി കുടി...,"കാരാപ്പുഴ പദ്ധതി: പകരം ഭൂമി നൽകിയില്ല, കുടിയിറ...",കണ്ഠൂര്: തുടര്ച്ചയായ മഴയെത്തുടര്ന്ന് കണ്ണൂര് വ...,['കണ്ണൂർ: തുടർച്ചയായി എത്തുന്ന മഴയും മണ്ണിടിച്...
1,കാബൂള്: തനിക്ക് പരിക്കൊന്നുമില്ലെന്നും പരിക്കി...,"എനിക്ക് പരിക്കില്ല, ടീമില് നിന്ന് ഒഴിവാക്കിയതാ...",അഫ്ഗാനിസ്ഥാന് പര്യടനത്തില് നിന്ന് വിരാട് കോഹ്ല...,['തിരുവല്ല: കാന്സര് ഇല്ലാത്ത രോഗിക്ക് കീമോതെറാ...
2,തിരുവനന്തപുരം: കേരള സർക്കാർ കായിക യുവജന കാര്യാ...,സ്കൂൾ കുട്ടികൾക്കായി ബാസ്ക്കറ്റ് ബോൾ പരിശീലന പ...,കേരള സർക്കാർ സ്പോൺസർ ചെയ്യുന്ന അടിസ്ഥാന തല ബാസ...,['സ്കൂൾതലം മുതൽ ബാസ്ക്കറ്റ്ബോളിൽ അന്താരാഷ്ട്ര ...
3,തിരുവനന്തപുരം: രാജ്യത്ത് ഗോഡ്സെ ക്ഷേത്രങ്ങള് വ...,ഗോ ഡ്സെ യ്ക്ക് ക്ഷേ ത്ര ങ്ങ ൾ നി ർ മ്മി ക്കു ...,കേരളത്തിൽ തീവ്രവാദ ബന്ധം ആരോപിച്ച് സിപിഎമ്മിന്...,['തിരുവനന്തപുരം: തീവ്രവാദ ബന്ധമുള്ള സിപിഎം പ്ര...
4,തിരുവനന്തപുരം : കാട്ടാക്കട പാറശ്ശാല നിയോജകമണ്ഡ...,മുടങ്ങിക്കിടന്ന മുഴുവന് പൊതുമരാമത്ത് പ്രവര്ത്ത...,കേരളത്തിലെ പൊതുഗതാഗത സംവിധാനത്തിന് ഗുണകരമായ നട...,['ആലപ്പുഴ: നിർമാണത്തിൽ കൃത്രിമം കാണിക്കാത്ത കര...


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

for col in ["article", "reference", "prediction", "contexts"]:
    df[col] = df[col].apply(clean_text)

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    scores = scorer.score(row["reference"], row["prediction"])

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

rouge1 = np.mean(rouge1_scores)
rouge2 = np.mean(rouge2_scores)
rougeL = np.mean(rougeL_scores)

print(rouge1, rouge2, rougeL)

100%|██████████| 970/970 [00:00<00:00, 4703.96it/s]

0.007628865979381443 0.0002577319587628866 0.007628865979381443


In [ ]:
references = [[ref] for ref in df["reference"].tolist()]
predictions = df["prediction"].tolist()

bleu = corpus_bleu(predictions, references).score

print("BLEU:", bleu)

BLEU: 3.673526562988939


In [ ]:
embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
semantic_scores = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_emb = embed_model.encode(str(row["prediction"]), convert_to_tensor=True)
    ref_emb = embed_model.encode(str(row["reference"]), convert_to_tensor=True)

    sim = util.cos_sim(pred_emb, ref_emb).item()
    semantic_scores.append(sim)

bertscore = np.mean(semantic_scores)

print("Semantic Similarity:", bertscore)

100%|██████████| 970/970 [00:30<00:00, 31.69it/s]

Semantic Similarity: 0.5195360004652253


In [ ]:
faithfulness_scores = []
context_precision_scores = []
context_recall_scores = []
hallucination_flags = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    article = row["article"]
    context = row["contexts"]
    prediction = row["prediction"]

    pred_emb = embed_model.encode(prediction, convert_to_tensor=True)
    article_emb = embed_model.encode(article, convert_to_tensor=True)
    context_emb = embed_model.encode(context, convert_to_tensor=True)

    faithfulness = util.cos_sim(pred_emb, article_emb).item()
    context_precision = util.cos_sim(pred_emb, context_emb).item()
    context_recall = util.cos_sim(context_emb, pred_emb).item()

    faithfulness_scores.append(faithfulness)
    context_precision_scores.append(context_precision)
    context_recall_scores.append(context_recall)

    if faithfulness < 0.75:
        hallucination_flags.append(1)
    else:
        hallucination_flags.append(0)

100%|██████████| 970/970 [00:34<00:00, 27.80it/s]


In [ ]:
faithfulness = np.mean(faithfulness_scores)
context_precision = np.mean(context_precision_scores)
context_recall = np.mean(context_recall_scores)
hallucination_rate = np.mean(hallucination_flags)

In [ ]:
hallucination_score = 1 - (
    0.4 * faithfulness +
    0.2 * context_precision +
    0.2 * context_recall +
    0.2 * bertscore
)

print("Faithfulness:", faithfulness)
print("Context Precision:", context_precision)
print("Context Recall:", context_recall)
print("Hallucination Score:", hallucination_score)
print("Hallucination Rate:", hallucination_rate)

Faithfulness: 0.5864259876982914
Context Precision: 0.5079593741671019
Context Recall: 0.5079593741671019
Hallucination Score: 0.45833865516079764
Hallucination Rate: 0.6969072164948453


In [ ]:
results = {
    "Model": "Sarvam",
    "Setting": "Pretrained+RAG",

    "ROUGE1": rouge1,
    "ROUGE2": rouge2,
    "ROUGEL": rougeL,
    "BLEU": bleu,
    "BERTScore": bertscore,

    "Faithfulness": faithfulness,
    "ContextPrecision": context_precision,
    "ContextRecall": context_recall,

    "HallucinationScore": hallucination_score,
    "HallucinationRate": hallucination_rate
}

metrics_df = pd.DataFrame([results])

metrics_df.to_csv(
    f"{BASE_DIR}/sarvam_rag_final_metrics.csv",
    index=False
)

display(metrics_df)

,Model,Setting,ROUGE1,ROUGE2,ROUGEL,BLEU,BERTScore,Faithfulness,ContextPrecision,ContextRecall,HallucinationScore,HallucinationRate
0,Sarvam,Pretrained+RAG,0.007629,0.000258,0.007629,3.673527,0.519536,0.586426,0.507959,0.507959,0.458339,0.696907


In [ ]:
df["Faithfulness"] = faithfulness_scores
df["ContextPrecision"] = context_precision_scores
df["ContextRecall"] = context_recall_scores
df["HallucinationFlag"] = hallucination_flags

df.to_csv(
    f"{BASE_DIR}/sarvam_rag_detailed_analysis.csv",
    index=False
)

print("Saved.")

Saved.


In [ ]:
import pandas as pd

rag_results = [
    {
        "Model": "BLOOMZ",
        "Setting": "Pretrained+RAG",

        "ROUGE1": 0.007800687285223367,
        "ROUGE2": 0.0,
        "ROUGEL": 0.007697594501718213,
        "BLEU": 0.218892648467054,
        "BERTScore": 0.7798730134963989,

        "Faithfulness": 0.7977745532989502,
        "ContextPrecision": 0.9969072164948454,
        "ContextRecall": 1.0,

        "HallucinationScore": 0.12553415357452086,
        "HallucinationRate": 0.0
    },

    {
        "Model": "mT5",
        "Setting": "Pretrained+RAG",

        "ROUGE1": 0.0010505645557191948,
        "ROUGE2": 0.0,
        "ROUGEL": 0.0010014727540500737,
        "BLEU": 0.37259275585198626,
        "BERTScore": 0.7824975252151489,

        "Faithfulness": 0.7870193719863892,
        "ContextPrecision": 1.0,
        "ContextRecall": 1.0,

        "HallucinationScore": 0.12869271975509908,
        "HallucinationRate": 0.0
    },

    {
        "Model": "mBART",
        "Setting": "Pretrained+RAG",

        "ROUGE1": 0.006734502875617106,
        "ROUGE2": 0.0,
        "ROUGEL": 0.006655515479658758,
        "BLEU": 0.16443164090688803,
        "BERTScore": 0.7900398373603821,

        "Faithfulness": 0.8228835463523865,
        "ContextPrecision": 0.6989690721649484,
        "ContextRecall": 1.0,

        "HallucinationScore": 0.17304481380993567,
        "HallucinationRate": 0.0
    }
]

rag_df = pd.DataFrame(rag_results)

display(rag_df)

,Model,Setting,ROUGE1,ROUGE2,ROUGEL,BLEU,BERTScore,Faithfulness,ContextPrecision,ContextRecall,HallucinationScore,HallucinationRate
0,BLOOMZ,Pretrained+RAG,0.007801,0.0,0.007698,0.218893,0.779873,0.797775,0.996907,1.0,0.125534,0.0
1,mT5,Pretrained+RAG,0.001051,0.0,0.001001,0.372593,0.782498,0.787019,1.000000,1.0,0.128693,0.0
2,mBART,Pretrained+RAG,0.006735,0.0,0.006656,0.164432,0.790040,0.822884,0.698969,1.0,0.173045,0.0


In [ ]:
sarvam_row = {
    "Model": "Sarvam",
    "Setting": "Pretrained+RAG",

    "ROUGE1": rouge1,
    "ROUGE2": rouge2,
    "ROUGEL": rougeL,
    "BLEU": bleu,
    "BERTScore": bertscore,

    "Faithfulness": faithfulness,
    "ContextPrecision": context_precision,
    "ContextRecall": context_recall,

    "HallucinationScore": hallucination_score,
    "HallucinationRate": hallucination_rate
}

rag_df = pd.concat([rag_df, pd.DataFrame([sarvam_row])], ignore_index=True)

display(rag_df)

,Model,Setting,ROUGE1,ROUGE2,ROUGEL,BLEU,BERTScore,Faithfulness,ContextPrecision,ContextRecall,HallucinationScore,HallucinationRate
0,BLOOMZ,Pretrained+RAG,0.007801,0.000000,0.007698,0.218893,0.779873,0.797775,0.996907,1.000000,0.125534,0.000000
1,mT5,Pretrained+RAG,0.001051,0.000000,0.001001,0.372593,0.782498,0.787019,1.000000,1.000000,0.128693,0.000000
2,mBART,Pretrained+RAG,0.006735,0.000000,0.006656,0.164432,0.790040,0.822884,0.698969,1.000000,0.173045,0.000000
3,Sarvam,Pretrained+RAG,0.007629,0.000258,0.007629,3.673527,0.519536,0.586426,0.507959,0.507959,0.458339,0.696907


In [ ]:
RAG_DIR = "/content/drive/MyDrive/FINAL_RAG_RESULTS"
os.makedirs(RAG_DIR, exist_ok=True)

rag_df.to_csv(
    f"{RAG_DIR}/final_rag_comparison.csv",
    index=False
)

print("Saved.")

Saved.


In [ ]:
import pandas as pd
import os

# Load existing 12-model comparison
master_df = pd.read_csv(
    "/content/drive/MyDrive/master_12_model_comparison.csv"
)

# RAG rows
rag_results = [
    {
        "Model": "BLOOMZ",
        "Setting": "Pretrained+RAG",
        "ROUGE1": 0.007800687285223367,
        "ROUGE2": 0.0,
        "ROUGEL": 0.007697594501718213,
        "BLEU": 0.218892648467054,
        "BERTScore": 0.7798730134963989,
        "Faithfulness": 0.7977745532989502,
        "ContextPrecision": 0.9969072164948454,
        "ContextRecall": 1.0,
        "HallucinationScore": 0.12553415357452086,
        "HallucinationRate": 0.0
    },

    {
        "Model": "mT5",
        "Setting": "Pretrained+RAG",
        "ROUGE1": 0.0010505645557191948,
        "ROUGE2": 0.0,
        "ROUGEL": 0.0010014727540500737,
        "BLEU": 0.37259275585198626,
        "BERTScore": 0.7824975252151489,
        "Faithfulness": 0.7870193719863892,
        "ContextPrecision": 1.0,
        "ContextRecall": 1.0,
        "HallucinationScore": 0.12869271975509908,
        "HallucinationRate": 0.0
    },

    {
        "Model": "mBART",
        "Setting": "Pretrained+RAG",
        "ROUGE1": 0.006734502875617106,
        "ROUGE2": 0.0,
        "ROUGEL": 0.006655515479658758,
        "BLEU": 0.16443164090688803,
        "BERTScore": 0.7900398373603821,
        "Faithfulness": 0.8228835463523865,
        "ContextPrecision": 0.6989690721649484,
        "ContextRecall": 1.0,
        "HallucinationScore": 0.17304481380993567,
        "HallucinationRate": 0.0
    }
]

rag_df = pd.DataFrame(rag_results)

# Merge
final_df = pd.concat([master_df, rag_df], ignore_index=True)

# Save
SAVE_DIR = "/content/drive/MyDrive/FINAL_BENCHMARK_RESULTS"
os.makedirs(SAVE_DIR, exist_ok=True)

final_df.to_csv(
    f"{SAVE_DIR}/master_15_model_comparison.csv",
    index=False
)

display(final_df)

,Model,Setting,ROUGE1,ROUGE2,ROUGEL,BLEU,BERTScore,Faithfulness,ContextPrecision,ContextRecall,HallucinationScore,HallucinationRate
0,Sarvam,ZeroShot,0.002168,0.000142,0.002208,0.802404,0.816132,0.854938,0.967010,1.0,0.101397,0.0
1,BLOOMZ,ZeroShot,0.013747,0.000000,0.013727,1.068298,0.795823,0.811640,0.992784,1.0,0.117623,0.0
2,mT5,ZeroShot,0.000295,0.000000,0.000295,0.025067,0.768236,0.763569,1.000000,1.0,0.140925,0.0
3,mBART,ZeroShot,0.014320,0.001718,0.014489,0.686449,0.809734,0.847780,0.898969,1.0,0.119147,0.0
4,Sarvam,TwoShot,0.000471,0.000000,0.000451,0.342049,0.816985,0.838780,1.000000,1.0,0.101091,0.0
5,BLOOMZ,TwoShot,0.009966,0.000000,0.009966,0.024048,0.773784,0.772429,0.991753,1.0,0.137921,0.0
6,mT5,TwoShot,0.004400,0.000395,0.004380,0.957940,0.834877,0.874746,1.000000,1.0,0.083126,0.0
7,mBART,TwoShot,0.006529,0.000000,0.006186,1.524800,0.802297,0.818707,0.824742,1.0,0.147110,0.0
8,mBART,FineTuned,0.037698,0.001718,0.037457,14.426935,0.885237,0.912175,1.000000,1.0,0.058083,0.0
9,mT5,FineTuned,0.032509,0.001031,0.032887,14.176625,0.879882,0.922249,0.998969,1.0,0.055330,0.0


In [ ]:
sarvam_row = {
    "Model": "Sarvam",
    "Setting": "Pretrained+RAG",
    "ROUGE1": rouge1,
    "ROUGE2": rouge2,
    "ROUGEL": rougeL,
    "BLEU": bleu,
    "BERTScore": bertscore,
    "Faithfulness": faithfulness,
    "ContextPrecision": context_precision,
    "ContextRecall": context_recall,
    "HallucinationScore": hallucination_score,
    "HallucinationRate": hallucination_rate
}

final_df = pd.concat([final_df, pd.DataFrame([sarvam_row])], ignore_index=True)

final_df.to_csv(
    f"{SAVE_DIR}/master_16_model_comparison.csv",
    index=False
)

display(final_df)

,Model,Setting,ROUGE1,ROUGE2,ROUGEL,BLEU,BERTScore,Faithfulness,ContextPrecision,ContextRecall,HallucinationScore,HallucinationRate
0,Sarvam,ZeroShot,0.002168,0.000142,0.002208,0.802404,0.816132,0.854938,0.967010,1.000000,0.101397,0.000000
1,BLOOMZ,ZeroShot,0.013747,0.000000,0.013727,1.068298,0.795823,0.811640,0.992784,1.000000,0.117623,0.000000
2,mT5,ZeroShot,0.000295,0.000000,0.000295,0.025067,0.768236,0.763569,1.000000,1.000000,0.140925,0.000000
3,mBART,ZeroShot,0.014320,0.001718,0.014489,0.686449,0.809734,0.847780,0.898969,1.000000,0.119147,0.000000
4,Sarvam,TwoShot,0.000471,0.000000,0.000451,0.342049,0.816985,0.838780,1.000000,1.000000,0.101091,0.000000
5,BLOOMZ,TwoShot,0.009966,0.000000,0.009966,0.024048,0.773784,0.772429,0.991753,1.000000,0.137921,0.000000
6,mT5,TwoShot,0.004400,0.000395,0.004380,0.957940,0.834877,0.874746,1.000000,1.000000,0.083126,0.000000
7,mBART,TwoShot,0.006529,0.000000,0.006186,1.524800,0.802297,0.818707,0.824742,1.000000,0.147110,0.000000
8,mBART,FineTuned,0.037698,0.001718,0.037457,14.426935,0.885237,0.912175,1.000000,1.000000,0.058083,0.000000
9,mT5,FineTuned,0.032509,0.001031,0.032887,14.176625,0.879882,0.922249,0.998969,1.000000,0.055330,0.000000
